# v3.10 world spec — the preparation world

*Agreed 5 September 2026. D1 accepted with the rig check reframed; the type-blind analysis folded
in; D2–D6 accepted as recommended. **Implemented**, pre-checks run. Findings from those pre-checks
are at the end, including two that need your call.*

**Why this world.** v3.9 closed the recipe world: a chain whose payoff arrives only at the end is
sparse, and a rare opportunity cannot generate the selection differential that would build the
approach behaviour making it less rare. The fix is not a bigger payoff or a shorter journey — both
were tried — but **a dense opportunity**. Here the fact to be learned sits on **every meal**, where
the agent already is. There is no approach behaviour to evolve, no carrying, no navigation: only
*which action to take on the food under my feet*.

---

## Cells and co-occupancy

Two contents only: **food A** and **food B**, one array pair, at most one type per cell (enforced at
spawn), spawning in drifting patches exactly as v3.1. **No items, no stations, no nuts.** Agents do
not block each other; occupancy is an observation channel only.

## Actions — eight, of which five are live in phase 1

`0–3` move N/S/E/W · `4` **eat** · `5,6,7` **prep_1 / prep_2 / prep_3**

In **phase 1** the three preparation logits are forced to −∞, so phase 1 is **exactly v3.1's five
actions** and its gate applies unchanged. **Only the action space changes at the switch** — no new
input channels, so the transition is one thing, not two.

| action | food A here | food B here | otherwise |
|---|---|---|---|
| **eat** | eat raw: safe → **+`food_value`**, **m = +1**; poison → **−`poison_value`**, **m = −1** | same | no-op, `move_cost` |
| **prep_k** | correct prep for A → **+`prep_value`**, **m = +1**; wrong → **−`prep_fail`**, **m = −1** | correct prep for B → **+`prep_value`**, **m = +1**; wrong → **−`prep_fail`**, **m = −1** | no-op, `move_cost` |

A preparation **consumes the food cell**, exactly as eating does.

**RESOLVED — D1 accepted, and the rig check is reframed.** The consequence is stronger than "noisy":
once the mapping is known, preparation pays 1.5 on *any* food, so the safe/poison fact becomes
**irrelevant** and a good learner stops eating raw entirely. Food learning is then *unselected*, and
`probe_adv` (food) decays for a **good reason** — not because the transition broke `H`.

So **rig check 2(a) is read in the first mapping era of phase 2 only** (steps 0–2000 after the
switch), when nobody knows the mapping and `eat` is still the right action. After that, **prep share
of meals by era** is the diagnostic: a learner that has the mapping should shift from `eat` to
`prep`, and that shift is itself an efficiency signature.

## Modulator events — the complete list

| event | m | energy |
|---|---|---|
| eat safe food | **+1** | +`food_value` |
| eat poison | **−1** | −`poison_value` |
| **correct preparation** | **+1** | +`prep_value` |
| **wrong preparation** | **−1** | −`prep_fail` |
| everything else | **0** | move / no-op `move_cost`, base metabolism, repair |

There is no silent event in this world: every action on food resolves immediately and two-sidedly.
The modulator is **not** the agent's own energy change in general; it is this table.

## Three levels on prep hit — and the type-blind floor

| level | value | what it means |
|---|---|---|
| chance | **1/3** | a random preparation |
| **type-blind** | **0.5** | "always `prep_k`" for a k useful in this era: right for one food type, wrong for the other, **no type knowledge at all**, EV +0.5/meal |
| full | **1.0** | the conjunction: the right preparation for each type |

With distinct mappings, exactly one preparation is useless in any era and each of the other two is
correct for one type. A type-blind policy therefore scores **0.5 within a favourable era** — but
**only 1/3 averaged across eras**, because k is useless in a third of mappings (EV −0.5/meal there).
**A genome beats chance only by tracking the era, not by holding one preparation.**

**A conjunction shows as BOTH types above 0.5**, not one at 1.0 and the other at 0. Per-type hit is
printed for every arm.

## The mapping

Each food type has exactly one correct preparation, so the mapping is a function
{A, B} → {prep_1, prep_2, prep_3}. Drawn at the switch and **redrawn every `prep_every = 2000`
steps**, independently of the safe/poison flip. Chance prep hit = **1/3**.

> **DECISION 2 — must the two food types map to *different* preparations?** If drawn independently,
> they collide one time in three, and a colliding mapping is solvable by learning a single
> preparation and never discriminating food type at all — which is not the task.
> **Recommendation: draw them distinct**, and redraw so that the new mapping differs from the old in
> at least one type (as `new_recipe` already does for the recipe). Chance stays 1/3 per prepared meal.

## Observation — unchanged, and unchanged across the switch

The v3.8/v3.9 layout, 60 inputs. Live: food A / food B / occupancy directional sums and here-values,
plus energy. Nothing tells the agent the mapping; nothing changes at the switch.

> **DECISION 3 — keep the 60-input layout with the chain channels permanently dead, or shrink to the
> ~21 live inputs?** **Recommendation: keep 60.** It costs nothing (dead inputs contribute exactly
> zero), it keeps the phase-1 gate directly comparable with v3.8's and v3.9's, and shrinking would
> change the network size at the same moment as everything else. The dead channels are named in the
> setup print so nobody mistakes them for live.

Hidden 24. No instinct wired: `scaffold_food=False`, `scaffold_chain=False`; `nav_dir` / `nav_here`
stay in the genome, reach nothing, and give the dead-gene drift scale.

## Densities

Food exactly as v3.1 (`spawn_per_patch` 3.0, 8 patches, radius 6, `food_rot` 0.005) — measured
standing cover ~43%. **There is no density to tune in this world**: the opportunity is every food
cell, so the readability criterion that failed three times in v3.9 is satisfied by construction.
Printed in setup for the record, not asserted as a band.

## Phases — one population, never rebuilt

**Phase 1:** 8000 steps, v3.1 unchanged, preparations masked. **Phase 2:** 8000 steps, preparations
live for the same population; agents, `H`, eligibility traces and the world all carry across
untouched. Mapping redrawn at the switch and every 2000 steps thereafter, so phase 2 holds four
whole mapping eras.

## Costs and values

v3.1 metabolism: `base_cost` 0.006 · `move_cost` 0.002 · `start_energy` 1.5 · `founder_energy` 3.0 ·
`repro_threshold` 3.0 · `repro_cost` 1.5 · `max_energy` 5.0 · `max_pop` 400 · `init_pop` 300 ·
`min_pop` 40 · food +0.7 / poison −0.5 · `flip_every` 300 · `eta_init` 0.2.

Preparation: **`prep_value` 1.5** · **`prep_fail` 0.5** · `prep_every` 2000 · K = 3.

Expected value of a preparation at chance: `1/3 × 1.5 − 2/3 × 0.5 = +0.167`, against `+0.1` for a
raw meal at a chance safe rate and `+0.7` for a known-safe one. Knowing the mapping is worth
**+1.5 per meal against +0.7** — and unlike v3.9, the opportunity arrives on **every meal**, so the
differential compounds over a lifetime instead of appearing half a time.

## Arms — five, three seeds

`random policy` · `fixed` · `scrambled` · `plastic (W2)` · `fixed + B (ceiling)`

`random policy` is uniform over the **available** actions — 5 in phase 1, 8 in phase 2 — so the
conditional null is **1/5** then **1/8** per action, and **3/8** for "any preparation". Exempt from
row 0: a random walker belongs at the population floor.

> **DECISION 4 — how does the `B` ceiling act?** In the recipe world B steered navigation and vetoed
> attempts. There is no navigation here; the only choice is which action to take on the food
> underfoot. **Recommendation: B is a hand-wired table over (food type, prep) updated with exact
> credit (`B[f,k] ← 0.7·B[f,k] + 0.3·outcome`), and when the agent stands on food and any entry for
> that type exceeds a threshold, the action is forced to `argmax_k B[f,k]`; otherwise the network
> chooses.** That is the honest analogue of v2's veto: a hand-wired policy override, not a hint. It
> remains a reference level, not a matched comparison, and the summary says so.

## Metrics

**prep hit rate**, event-weighted (correct preps / preps; chance **1/3**) · **P(prep | on food)**
against the null, and per-prep P(prep_k | on food) · **`probe_adv` (prep)**, within-agent, built
like the recipe probe: on a synthetic "food type f here" observation, the logit of the correct prep
minus the mean of the other two, with learned synapses minus innate · **hit-in-life curve** (prep hit
by prep number in an agent's life) and **since-remap curve** (by preps since the last remap) ·
`safe_rate`, raw-meal count and **`probe_adv` (food)** through phase 2 as the rig check · **prepared
meals per life** · `eta2` / `lam2` / `eta1` against `fixed` · unwired `nav_dir` / `nav_here` as drift
scale · transition table at 500-step bins across the switch · populations and injections per phase.

## Stopping rule

Event-weighted, per phase. Margin **0.03**, seeds **3/3**. A positive on any row gets seeds 3–4
before it is called.

| # | row | reading | attributing arm |
|---|---|---|---|
| **0** | uninterpretable per phase: pop < 80 or injections > 0 | exclude and name it. `random policy` exempt | — |
| **1a** | **phase-1 gate = v3.1** — safe rate `plastic` − `fixed` ≥ 0.03, `probe_adv` (food) ≥ 1.0 | **STOP ROW.** If phase 1 is not v3.1, nothing below is read. Row-0 fallback: where `fixed` phase 1 is excluded, that seed reads against v3.1's published range, conservative end 0.56 | `fixed`, or v3.1's published range |
| **1b** | **mapping gate** — `fixed` prep hit **≤ 0.55** in 3/3 | genes **may** hold a type-blind preparation (0.5); they must not track the **conjunction**. `fixed`'s per-type hit is printed: one type high and the other near 0 is type-blind and allowed; **both above 0.5 in `fixed`** would be genes holding the conjunction. If it fires, shorten `prep_every` toward the flip period, **judged against `fixed` only** | `fixed`, per-type |
| **2** | **rig checks** — (a) food learning survives the switch, **read on safe rate in the first mapping era** (`plastic` − `fixed` ≥ 0.03); the corrected `probe_adv` (food) in the **first 500 steps** after the switch is **corroborating only**, and the **eat → prep shift timing** (prep share in 250-step bins) plus **prep share by era** are printed as diagnostics; (b) the opportunity exists: **prepared meals per life ≥ 3 in `fixed`**; (c) P(prep \| on food) against the per-phase null | if (a) or (b) fails the rig is broken: stop, diagnose, no learner claim. **Why the reframe:** the v3.1 probe form subtracts the best *other* action, and with three preparations live that term moves with food type, so it measured preparation preference, not food learning. The probe is corrected (`eat` minus the mean move logit), but a *decaying* probe after the first era is expected behaviour — once the mapping is known, preparation pays 1.5 on any food and the safe/poison fact stops mattering | `fixed`, `random policy` |
| **3** | **the conjunction.** `plastic` − `fixed` ≥ 0.03 **and** `plastic` − `scrambled` ≥ 0.03 in 3/3 on prep hit; `probe_adv` (prep) > 0; **one within-life signature**; **no abstention** (prepared meals not below 0.8× `fixed`) | **a positive means clearing the type-blind floor.** Read the per-type split: the conjunction is **both types above 0.5**. **`plastic` − `scrambled` on hit rate is a survivorship-contaminated contrast** — agents whose random `H` happens to help live longer, enriching the standing population with nothing learned. It stays required, but the **attribution** is carried by the within-agent lines: `probe_adv` (prep) and row 3a | `scrambled` carries the claim; `probe_adv` is within-agent; abstention read first |
| **3a** | **survivorship diagnostics (required).** (i) **survivor curve** — hit on preparations 1–5 vs 6–10 over agents that reached 10 preparations: **rising in `plastic`, flat in `scrambled`**. Every agent counted contributes both halves of its own curve, so a rise is the same individuals later in their own lives, not a different sample. (ii) **first-preparation hit** — the agent has learned nothing, so this reads the innate policy the standing population carries | a `plastic` − `scrambled` gap on hit rate with a **flat** survivor curve is enrichment, not learning, and row 3 is **not** called positive on it. Note `probe_adv` is computed over the **living** and so is itself partly survivorship-selected; the survivor curve is not | within-agent |
| **3b** | **knockout** — late `plastic` and `scrambled` genomes replayed with `eta_scale = 0` for 3000 steps in a **fresh world seed**: same brains, no learning | if the standing advantage lives in `H`, **both** fall to the type-blind floor. What then separates them is the survivor curve, which is learning rather than luck. A `plastic` genome that keeps its advantage with learning off had it in the **genome**, not in `H` | within-genome |
| **4** | gene rows: `eta2`, `lam2`, `eta1` against `fixed` and `scrambled`; unwired scaffold genes as drift scale | corroborating only | — |
| **5** | if row 3 is null with rows 1–2 clean | the first earned statement about the learner's limit, on a dense, immediate, two-sided task. Spend the one rule-form change there | — |

**Pre-registered prediction, recorded before the run:** `fixed` sits near **0.5** and **crashes in
the third of eras where its k goes useless** (EV −0.5/meal); `plastic (W2)` clears **0.5 on both
types** and **recovers across remaps**.

> **DECISION 5 — the within-life signature is harder here than it looks.** A mapping era is 2000
> steps and a generation is ~200, so most agents live inside a single era and never see a remap.
> `hit_old` > `hit_young` and the hit-in-life curve therefore measure learning *within* an era, which
> is what we want — but an agent born mid-era inherits nothing and must learn from its own
> preparations. **Recommendation: keep `prep_every` at 2000 and read the since-remap curve
> population-wide**, which is where a remap's cost and recovery show. If gate 1b fires and
> `prep_every` shortens, both curves become more informative, not less.

> **DECISION 6 — the semantics test needs the mapping in the loop.** The v3.9 test enumerated
> (action, cell content, inventory). Here the third dimension is the **mapping**.
> **Recommendation: enumerate every (action ∈ {eat, prep_1..3}, food ∈ {none, A, B}, mapping ∈ all
> distinct assignments) row** — 4 × 3 × 6 = 72 rows plus the four move/no-op rows — constructing the
> cell, calling one `resolve_action`, and asserting energy delta, `m`, event and that the food cell
> was consumed. It runs well under a second and it is the test that would catch a mapping applied to
> the wrong food type.

## Also carried over unchanged

The learning rule and its genes; the staging mechanism; event-weighted aggregation per phase; the
conditional-null convention (measured null printed beside the analytic one); the learning-rule unit
test in the setup cell; QUICK mode; Colab-only with `sim.py` and `analysis.py` uploaded alongside;
per-run pickling with resume instructions.


---

# Findings from the pre-checks (1 seed, 3000-step phases, every arm)

**The world is readable — the thing v3.9 never achieved.** Prepared meals per life: `fixed` **16.5**,
`plastic` **19.7**, ceiling **62.2**, null **6.6**, against a criterion of ≥ 3. There is no density
to tune and no approach behaviour to evolve.

| arm | P2 pop | prep hit | hit \| A | hit \| B | prep/life | probe_adv (prep) |
|---|---|---|---|---|---|---|
| random policy | 117 | **0.334** | 0.337 | 0.330 | 6.6 | — |
| fixed | 338 | **0.500** | 0.740 | 0.256 | 16.5 | — |
| scrambled | 399 | 0.722 | 0.553 | 0.840 | 15.8 | **0.342** |
| plastic (W2) | 399 | 0.833 | 0.934 | 0.692 | 19.7 | **5.624** |
| fixed + B (ceiling) | 399 | 0.886 | 0.879 | 0.894 | 62.2 | — |

`random policy` lands on chance and **`fixed` lands on exactly the type-blind floor, 0.500, with
A 0.740 / B 0.256** — the predicted signature, and gate 1b is clear at ≤ 0.55.

### Three things that need your call

**1. `scrambled` sits well above the type-blind floor (0.722, both types above 0.5).** The control is
showing what looks like conjunction knowledge. Its **within-agent probe is 0.342 against plastic's
5.624**, a 16× gap, so the population-level hit rate carries a **survivorship** component the probe
does not: agents whose random `H` happens to favour the correct preparation live longer, so the
standing population is enriched for lucky `H` without anything being learned. Row 3 already requires
both lines, which is the right design — but the hit-rate margin against `scrambled` is partly a
survivorship comparison and should be read that way.

**2. Rig check 2(a) fails as specified, and I found the instrument at fault first.** The v3.1 food
probe is `logit(eat)` minus the **best other action** — and with three preparations live, that term
moves with food type, so the probe was measuring preparation preference. Fixed: the probe is now
`logit(eat)` against the **move** logits, which are food-type-independent. That raised the first-era
value from 0.040 to **0.196** — still far below the ≥ 1.0 criterion. The cause is that the eat→prep
shift happens **inside** the first era (prep share 0.836 in era 1), so the window the criterion
assumes is shorter than 2000 steps. Safe rate does hold there: `plastic` **0.599** against `fixed`
0.489. **Proposal:** read 2(a) on **safe rate in the first era** (`plastic` − `fixed` ≥ 0.03) with the
probe corroborating, or read the probe in the first ~500 steps after the switch. I have not measured
the 500-step version.

**3. Populations sit at the 400 cap** in three of five arms. At a hard cap births become a queue
rather than differential fecundity, which blunts selection — the same issue that cost a v3.6 tuning
pass. Raising `max_pop` is the fix; it is a spec change and yours to make.


## 1. Setup

The sim and the analysis are imported, not inlined. This cell fails loudly if either file is missing.

In [ ]:
# --- Colab check -------------------------------------------------------------
# Upload sim.py and analysis.py next to this notebook (Files pane, or run:
#     from google.colab import files; files.upload()
# and pick both).  Nothing else is needed: pure numpy + matplotlib.
import os, sys

missing = [f for f in ("sim.py", "analysis.py") if not os.path.exists(f)]
if missing:
    raise SystemExit(f"missing {missing} in {os.getcwd()} -- upload them next to this notebook")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import numpy as np
import sim, analysis as A

print("numpy", np.__version__)
print("observation size:", sim.N_IN, " actions:", sim.N_ACTIONS,
      " (eat =", sim.EAT, ", preps =", sim.PREP0, "..", sim.PREP0 + sim.N_PREPS - 1, ")",
      " hidden:", A.WORLD["hidden"], " chance recipe hit:", round(A.CHANCE, 3))
print("\nworld (v3.1 metabolism throughout; the chain from v3.6):")
for k, v in A.WORLD.items():
    print(f"  {k:<20} {v}")
print("\n--- world-semantics self-test (every row of the action x cell table) ---")
if not sim.world_semantics_selftest():
    raise SystemExit("the world does not match the spec; nothing below is meaningful")

print("\n--- density ---")
print("no density to assert: the opportunity is every food cell")

print("\n--- learning-rule self-test ---")
print("one agent, one fixed observation, one chosen action; action_noise = 0 so act() is")
print("deterministic and the logit checked belongs to the action that laid the trace.")
if not sim.learning_rule_selftest():
    raise SystemExit("the learning rule is not behaving; nothing below is meaningful")

print("\nconditions:")
for name, spec in A.VARIANTS.items():
    ph = " -> ".join(f"{p['n_steps']} steps chain={p['chain']}" for p in spec["phases"])
    print(f"  {name:<24} {ph}")
    print(f"  {'':<24} {({k: v for k, v in spec['kw'].items() if k not in A.WORLD})}")


## 2. Run

**Runtime.** A staged run is 16000 steps — twice a v3.6 run — and `max_pop` is now **800** (raised from 400 so `plastic` is not sitting on the cap), so cost scales with the standing population rather than with steps alone. Measured wall clock is printed by the acceptance run in the reading cell; budget from that number × 15 for the 5 × 3 grid, and expect Colab CPU to be slower than a local box.

The cell checkpoints to `results_v3_10.pkl` after every run, so a dropped session costs one run. To resume, reload the pickle and re-run only the missing seeds, then merge.

**Seeds 3–4 are held in reserve.** A positive on any row gets them before it is called. To append them later, with the same three files present:

```python
import pickle
base = pickle.load(open("results_v3_10.pkl", "rb"))
more = A.run_experiment(seeds=[3, 4], save_path="results_seeds34.pkl")
for k in base:
    base[k] += more[k]          # seed order stays [0,1,2,3,4]
pickle.dump(base, open("results_v3_10_all.pkl", "wb"))
results = base
```

The seed criterion adapts automatically: `min(4, n_seeds)`, so 3 seeds reads 3/3 and 5 seeds reads 4/5.

In [ ]:
# QUICK = True   exercises every cell below: 1 seed, 1500-step phases (3000 total), and
#                recipe_every dropped to 500 so recipe changes actually occur in phase 2.
#                A smoke test, NOT a result -- 1500 steps is only a handful of generations,
#                so row 0 will flag conditions as uninterpretable and row 1 will not reproduce.
# QUICK = False  the real experiment: 5 arms x 3 seeds, 8000-step phases (16000 total).
QUICK = True

if QUICK:
    SEEDS, PHASE_STEPS, OVERRIDES = [0], 1500, dict(recipe_every=500)
else:
    SEEDS, PHASE_STEPS, OVERRIDES = [0, 1, 2], 8000, {}      # seeds 3-4 held in reserve

# Pickled after every run, so a dropped Colab session costs one run rather than the lot.
# To resume:  import pickle; results = pickle.load(open("results_v3_10.pkl", "rb"))
results = A.run_experiment(seeds=SEEDS, phase_steps=PHASE_STEPS,
                           save_path="results_v3_10.pkl", **OVERRIDES)


## 3. The printed summary

Phase 1 and phase 2 blocks separately, then per-seed values, then the decision numbers in the order of the table above.

**Read row 1 first — it is a stop condition.** If phase 1 does not reproduce v3.1, nothing in phase 2 is readable.

**Reading a QUICK run:** a smoke test, not a result. Phases of 1500 steps are a handful of generations, so row 0 will flag conditions and row 1 will not reproduce. Read it only to confirm every cell produces the output it should.

In [ ]:
A.summary(results)


## 4. The transition

500-step bins across 2000 steps either side of the chain switching on. This is where a collapse becomes visible, and where `eta2`/`lam2` either hold or start falling as they did in v3.6.

In [ ]:
A.transition_table(results, bin_size=500, span=2000)


## 5. Curves

`meal` is v3.1's within-life food curve, read on **phase 1** — the positive control. `att` and `rec` are the recipe curves, read on **phase 2**; row 4 requires `att` to rise (or `hit_old` > `hit_young`).

In [ ]:
A.curves(results, "meal", "meal number in an agent's life")        # phase 1
A.curves(results, "att",  "attempt number in an agent's life")     # phase 2
A.curves(results, "rec",  "attempts since the last recipe change")  # phase 2


## 6. Plots

Solid black line = the chain switches on; dashed = recipe change.

In [ ]:
A.plot_results(results)


## 8. Reading it

**Rows 1a and 2 are stop conditions.**

1. **Row 1a** — phase 1 must be v3.1. If not, nothing below is readable.
2. **Row 1b** — `fixed` must not hold the *conjunction*. A type-blind 0.5 is allowed; read the
   per-type split to tell them apart.
3. **Row 2** — food learning in the first mapping era **read on safe rate** (`plastic` − `fixed`
   ≥ 0.03), with the corrected `probe_adv` (food) in the first 500 steps corroborating; then
   prepared meals per life ≥ 3 in `fixed`, and P(prep | on food) against the null. The
   **eat → prep shift timing** and **prep share by era** are printed as diagnostics, not gates.
   A probe that decays *after* the first era is expected: once the mapping is known, preparation
   pays on any food and the safe/poison fact stops mattering.
4. **Row 3** — the abstention line first, then the two result lines, then **the per-type split**:
   a conjunction is *both* types above 0.5, not one at 1.0 and one at 0.
5. **Rows 3a and 3b — survivorship.** `plastic` − `scrambled` on hit rate is a *contaminated*
   contrast: agents whose random `H` happens to help live longer, enriching the standing
   population with nothing learned. It stays required, but the attribution is carried by the
   within-agent lines. The **survivor curve** (preps 1–5 vs 6–10 over agents that reached 10)
   must **rise in `plastic` and stay flat in `scrambled`** — every agent counted contributes both
   halves of its own curve, so a rise is the same individuals later in their own lives. The
   **knockout** replays late genomes with `eta_scale = 0` in a fresh world: if the advantage lives
   in `H`, both arms fall to the type-blind floor. Note `probe_adv` is computed over the *living*
   and so is itself partly survivorship-selected; the survivor curve is not. The pre-checks had
   `scrambled` at 0.722 on hit rate but only 0.342 on the probe, against plastic's 5.624 — that
   gap is what these rows are here to adjudicate.
6. **Row 5** — a null here, with rows 1–2 clean, is the first earned statement about the learner's
   limit: no approach behaviour required, credit immediate and two-sided, opportunity on every meal.
